In [1]:
# plot_coldod_pinball_ladder.py

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# =========================
# 1. Paths
# =========================
# 改成你的项目根目录
PROJECT_ROOT = Path(r"E:\NetworkOptimization\pythonProject1")

# final combined strategic + existing results
RESULT_FILE = (
    PROJECT_ROOT
    / "Data/10_experiments/strategic_cold_od_baselines_v1_notebook/east_plus_gulf"
    / "leaderboard_test_strategic_plus_existing.csv"
)

# 如果你把结果 CSV 和脚本放在同一个目录，可自动 fallback
if not RESULT_FILE.exists():
    RESULT_FILE = Path("leaderboard_test_strategic_plus_existing.csv")

OUT_DIR = Path("figures")
OUT_DIR.mkdir(parents=True, exist_ok=True)

OUT_PNG = OUT_DIR / "fig_coldod_pinball.png"
OUT_PDF = OUT_DIR / "fig_coldod_pinball.pdf"
OUT_CSV = OUT_DIR / "data_fig_coldod_pinball.csv"


# =========================
# 2. Helpers
# =========================
def pick_one(df: pd.DataFrame, label: str, **filters) -> pd.Series:
    """Select exactly one row using strict filters."""
    sub = df.copy()
    for col, val in filters.items():
        sub = sub[sub[col].astype(str) == str(val)]

    if len(sub) != 1:
        cols = ["source", "model", "checkpoint", "seed", "split", "n_rows", "pinball_mean"]
        available = sub[cols].head(20).to_string(index=False) if len(sub) else "NO MATCH"
        raise ValueError(
            f"[{label}] expected exactly 1 row, got {len(sub)}.\n"
            f"Filters: {filters}\n"
            f"Matches:\n{available}"
        )
    return sub.iloc[0]


def assert_close(name: str, actual: float, expected: float, tol: float = 0.03) -> None:
    if abs(actual - expected) > tol:
        raise AssertionError(
            f"{name}: actual={actual:.6f}, expected={expected:.6f}, tol={tol}"
        )


# =========================
# 3. Load and select rows
# =========================
df = pd.read_csv(RESULT_FILE)

# Strict Cold-OD test rows only
df = df[(df["split"] == "test") & (df["n_rows"] == 1057)].copy()

rows_spec = [
    {
        "display": "ColdOD-MLP",
        "group": "Old baseline",
        "filters": {
            "model": "coldod_mlp_residual_prior",
            "checkpoint": "deduplicated_reference",
            "seed": "reference",
        },
        "expected_pinball": 205.925056,
    },
    {
        "display": "Numeric MLP\n(no zone ID)",
        "group": "Strategic controls",
        "filters": {
            "model": "numeric_mlp_no_id",
            "checkpoint": "best_val_pinball",
            "seed": "seed_ensemble",
        },
        "expected_pinball": 179.179627,
    },
    {
        "display": "STID-style\nID MLP",
        "group": "Strategic controls",
        "filters": {
            "model": "id_mlp_stid",
            "checkpoint": "best_val_pinball",
            "seed": "seed_ensemble",
        },
        "expected_pinball": 186.503521,
    },
    {
        "display": "Zone-ID-only\nMLP",
        "group": "Strategic controls",
        "filters": {
            "model": "zone_id_only_mlp",
            "checkpoint": "best_val_pinball",
            "seed": "seed_ensemble",
        },
        "expected_pinball": 194.110827,
    },
    {
        "display": "GraphSAGE-\nColdOD",
        "group": "Graph models",
        "filters": {
            "model": "graphv2_graphsage",
            "checkpoint": "deduplicated_reference",
            "seed": "reference",
        },
        "expected_pinball": 130.804053,
    },
    {
        "display": "HGT-\nColdOD",
        "group": "Graph models",
        "filters": {
            "model": "graphv2_hgt",
            "checkpoint": "deduplicated_reference",
            "seed": "reference",
        },
        "expected_pinball": 128.830055,
    },
]

plot_rows = []
for spec in rows_spec:
    row = pick_one(df, spec["display"], **spec["filters"])
    pin = float(row["pinball_mean"])
    assert_close(spec["display"], pin, spec["expected_pinball"])
    plot_rows.append(
        {
            "group": spec["group"],
            "model": spec["display"],
            "pinball": pin,
            "source": row["source"],
            "checkpoint": row["checkpoint"],
            "seed": row["seed"],
        }
    )

plot_df = pd.DataFrame(plot_rows)
plot_df.to_csv(OUT_CSV, index=False)


# =========================
# 4. Plot
# =========================
plt.rcParams.update(
    {
        "font.family": "DejaVu Sans",
        "font.size": 10,
        "axes.titlesize": 11,
        "axes.labelsize": 10,
        "xtick.labelsize": 9,
        "ytick.labelsize": 9,
    }
)

fig, ax = plt.subplots(figsize=(7.2, 3.8), dpi=300)

y = np.arange(len(plot_df))
bars = ax.barh(y, plot_df["pinball"])

ax.set_yticks(y)
ax.set_yticklabels(plot_df["model"])
ax.invert_yaxis()

ax.set_xlabel("Pinball loss (lower is better)")
ax.set_title("Cold-OD Pinball Ladder")

# clean academic style
ax.grid(axis="x", linestyle="--", alpha=0.3)
ax.set_axisbelow(True)
for spine in ["top", "right", "left"]:
    ax.spines[spine].set_visible(False)

# numeric labels
xmax = plot_df["pinball"].max()
for bar, val in zip(bars, plot_df["pinball"]):
    ax.text(
        val + xmax * 0.015,
        bar.get_y() + bar.get_height() / 2,
        f"{val:.2f}",
        va="center",
        ha="left",
        fontsize=9,
    )

# Add a concise improvement annotation
numeric_pinball = plot_df.loc[plot_df["model"].str.contains("Numeric"), "pinball"].iloc[0]
hgt_pinball = plot_df.loc[plot_df["model"].str.contains("HGT"), "pinball"].iloc[0]
reduction = (numeric_pinball - hgt_pinball) / numeric_pinball * 100

ax.text(
    hgt_pinball,
    len(plot_df) - 0.45,
    f"{reduction:.1f}% lower than Numeric MLP",
    ha="left",
    va="center",
    fontsize=8.5,
)

ax.set_xlim(0, xmax * 1.22)

fig.tight_layout()
fig.savefig(OUT_PNG, dpi=300, bbox_inches="tight")
fig.savefig(OUT_PDF, bbox_inches="tight")
plt.close(fig)

print(f"Saved: {OUT_PNG}")
print(f"Saved: {OUT_PDF}")
print(f"Saved source data: {OUT_CSV}")

Saved: figures\fig_coldod_pinball.png
Saved: figures\fig_coldod_pinball.pdf
Saved source data: figures\data_fig_coldod_pinball.csv


In [2]:
# plot_cvar_toprisk_regret.py

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# =========================
# 1. Paths
# =========================
# 改成你的项目根目录
PROJECT_ROOT = Path(r"E:\NetworkOptimization\pythonProject1")

RESULT_FILE = (
    PROJECT_ROOT
    / "Data/10_experiments/strategic_cold_od_baselines_v1_notebook/east_plus_gulf"
    / "cvar_toprisk_screening_strategic_plus_existing.csv"
)

# 如果你把结果 CSV 和脚本放在同一个目录，可自动 fallback
if not RESULT_FILE.exists():
    RESULT_FILE = Path("cvar_toprisk_screening_strategic_plus_existing.csv")

OUT_DIR = Path("figures")
OUT_DIR.mkdir(parents=True, exist_ok=True)

OUT_PNG = OUT_DIR / "fig_cvar_regret_top10.png"
OUT_PDF = OUT_DIR / "fig_cvar_regret_top10.pdf"
OUT_CSV = OUT_DIR / "data_fig_cvar_regret_top10.csv"


# =========================
# 2. Helpers
# =========================
def pick_one(df: pd.DataFrame, label: str, **filters) -> pd.Series:
    """Select exactly one row using strict filters."""
    sub = df.copy()
    for col, val in filters.items():
        sub = sub[sub[col].astype(str) == str(val)]

    if len(sub) != 1:
        cols = [
            "source",
            "model",
            "checkpoint",
            "seed",
            "score_type",
            "top_fraction",
            "k",
            "normalized_regret",
            "oracle_overlap",
        ]
        available = sub[cols].head(20).to_string(index=False) if len(sub) else "NO MATCH"
        raise ValueError(
            f"[{label}] expected exactly 1 row, got {len(sub)}.\n"
            f"Filters: {filters}\n"
            f"Matches:\n{available}"
        )
    return sub.iloc[0]


def assert_close(name: str, actual: float, expected: float, tol: float = 0.0008) -> None:
    if abs(actual - expected) > tol:
        raise AssertionError(
            f"{name}: actual={actual:.6f}, expected={expected:.6f}, tol={tol}"
        )


# =========================
# 3. Load and filter strict top-10% Cold-OD decision scope
# =========================
df = pd.read_csv(RESULT_FILE)

# Use strict top-10% setting from paper: n=1057, k=106
df = df[
    (np.isclose(df["top_fraction"].astype(float), 0.10))
    & (df["k"].round().astype(int) == 106)
].copy()

rows_spec = [
    {
        "display": "GraphSAGE-\nColdOD",
        "run": "ensemble",
        "filters": {
            "model": "graphv2_graphsage",
            "checkpoint": "deduplicated_reference",
            "seed": "reference",
            "score_type": "predicted_quantile_risk",
        },
        "expected_regret": 0.036467,
    },
    {
        "display": "HGT TruckRail+\nTopo",
        "run": "best-run",
        "filters": {
            "model": "hgt_truck_rail_plus_topology_features",
            "checkpoint": "best_val_q75",
            "seed": "42",
            "score_type": "predicted_quantile_risk",
        },
        "expected_regret": 0.038063,
    },
    {
        "display": "HGT-\nColdOD",
        "run": "ensemble/ref.",
        "filters": {
            "model": "graphv2_hgt",
            "checkpoint": "deduplicated_reference",
            "seed": "reference",
            "score_type": "predicted_quantile_risk",
        },
        "expected_regret": 0.047352,
    },
    {
        "display": "Zone-ID-only\nMLP",
        "run": "ensemble",
        "filters": {
            "model": "zone_id_only_mlp",
            "checkpoint": "best_val_pinball",
            "seed": "seed_ensemble",
            "score_type": "predicted_quantile_risk",
        },
        "expected_regret": 0.054198,
    },
    {
        "display": "STID-style\nID MLP",
        "run": "ensemble",
        "filters": {
            "model": "id_mlp_stid",
            "checkpoint": "best_val_q75",
            "seed": "seed_ensemble",
            "score_type": "predicted_quantile_risk",
        },
        "expected_regret": 0.057171,
    },
    {
        "display": "Numeric MLP\n(no zone ID)",
        "run": "ensemble",
        "filters": {
            "model": "numeric_mlp_no_id",
            "checkpoint": "best_val_pinball",
            "seed": "seed_ensemble",
            "score_type": "predicted_quantile_risk",
        },
        "expected_regret": 0.093187,
    },
    {
        "display": "ColdOD-\nMLP",
        "run": "reference",
        "filters": {
            "model": "coldod_mlp_residual_prior",
            "checkpoint": "deduplicated_reference",
            "seed": "reference",
            "score_type": "predicted_quantile_risk",
        },
        "expected_regret": 0.133011,
    },
    {
        "display": "Cold origin/\ndest prior",
        "run": "baseline",
        "filters": {
            "model": "cold_origin_dest_prior",
            "checkpoint": "raw",
            "seed": "prior",
            "score_type": "predicted_quantile_risk",
        },
        "expected_regret": 0.324164,
    },
    {
        "display": "Cold-prior\nq75 rank",
        "run": "heuristic",
        "filters": {
            "model": "cold_prior_q75_rank",
            "checkpoint": "score_only",
            "seed": "baseline",
            "score_type": "cold_prior_q75_rank",
        },
        "expected_regret": 0.336248,
    },
    {
        "display": "Random\nrank",
        "run": "heuristic",
        "filters": {
            "model": "random_rank",
            "checkpoint": "mean_100_runs",
            "seed": "random",
            "score_type": "random",
        },
        "expected_regret": 0.427117,
    },
    {
        "display": "Demand-volume\nrank",
        "run": "heuristic",
        "filters": {
            "model": "demand_volume_rank",
            "checkpoint": "score_only",
            "seed": "baseline",
            "score_type": "demand_volume_rank",
        },
        "expected_regret": 0.627442,
    },
]

plot_rows = []
for spec in rows_spec:
    row = pick_one(df, spec["display"], **spec["filters"])
    regret = float(row["normalized_regret"])
    overlap = float(row["oracle_overlap"])
    assert_close(spec["display"], regret, spec["expected_regret"])

    plot_rows.append(
        {
            "model": spec["display"],
            "run": spec["run"],
            "normalized_regret": regret,
            "oracle_overlap": overlap,
            "source": row["source"],
            "checkpoint": row["checkpoint"],
            "seed": row["seed"],
            "score_type": row["score_type"],
        }
    )

plot_df = pd.DataFrame(plot_rows)
plot_df.to_csv(OUT_CSV, index=False)


# =========================
# 4. Plot
# =========================
plt.rcParams.update(
    {
        "font.family": "DejaVu Sans",
        "font.size": 10,
        "axes.titlesize": 11,
        "axes.labelsize": 10,
        "xtick.labelsize": 9,
        "ytick.labelsize": 9,
    }
)

fig, ax = plt.subplots(figsize=(7.2, 5.2), dpi=300)

y = np.arange(len(plot_df))
bars = ax.barh(y, plot_df["normalized_regret"])

ax.set_yticks(y)
ax.set_yticklabels(plot_df["model"])
ax.invert_yaxis()

ax.set_xlabel("Normalized regret (lower is better)")
ax.set_title("Top-10% Decision-Aware OD-Risk Screening")

# clean academic style
ax.grid(axis="x", linestyle="--", alpha=0.3)
ax.set_axisbelow(True)
for spine in ["top", "right", "left"]:
    ax.spines[spine].set_visible(False)

xmax = plot_df["normalized_regret"].max()
for bar, val, ov in zip(bars, plot_df["normalized_regret"], plot_df["oracle_overlap"]):
    ax.text(
        val + xmax * 0.012,
        bar.get_y() + bar.get_height() / 2,
        f"{val:.3f}",
        va="center",
        ha="left",
        fontsize=8.8,
    )

ax.set_xlim(0, xmax * 1.18)

fig.tight_layout()
fig.savefig(OUT_PNG, dpi=300, bbox_inches="tight")
fig.savefig(OUT_PDF, bbox_inches="tight")
plt.close(fig)

print(f"Saved: {OUT_PNG}")
print(f"Saved: {OUT_PDF}")
print(f"Saved source data: {OUT_CSV}")

Saved: figures\fig_cvar_regret_top10.png
Saved: figures\fig_cvar_regret_top10.pdf
Saved source data: figures\data_fig_cvar_regret_top10.csv
